In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [2]:
import subprocess
import sys
import os

print("Starting consolidated dependency installation...")

# Upgrade pip first to ensure it's up-to-date
try:
    print("Upgrading pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    print("pip upgraded successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error upgrading pip: {e}")
    print("Continuing with installation, but consider manually updating pip if issues persist.")

# Clear pip cache to ensure fresh downloads
try:
    print("Clearing pip cache...")
    subprocess.check_call([sys.executable, "-m", "pip", "cache", "purge"])
    print("Pip cache cleared.")
except subprocess.CalledProcessError as e:
    print(f"Error clearing pip cache: {e}")

# This subprocess setup is crucial for fixing Google Colab's common `numpy` conflicts.
# The `numpy.dtype size changed` error often indicates binary incompatibility
# with pre-installed PyTorch/TorchVision when `numpy` is implicitly updated by other packages.
# By installing all core dependencies here and *avoiding* explicit `numpy` and `scipy`
# uninstallation/installation, we allow pip to resolve compatibility and rely on
# Colab's default `numpy`/`scipy` that are compatible with its `Torch`/`TorchVision` setup.
packages_to_install = [
    "albumentations==2.0.8", # Updated to the version installed later in the notebook
    "grad-cam==1.5.5", # Pinning to the version that was installed later in the notebook
    "ttach",
    "opencv-python", # Explicitly install opencv-python as used with cv2
    "matplotlib", # Explicitly install matplotlib as used for plotting
    "gradio", # Added gradio for the UI part
    "rembg[cpu]" # Re-adding rembg[cpu] to ensure onnxruntime backend is installed
]

try:
    print(f"Installing {', '.join(packages_to_install)}...")
    # Use --upgrade to ensure they are the latest available if already present
    # Use --no-cache-dir to ensure fresh download, combined with cache purge earlier
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir"] + packages_to_install)
    print("All specified dependencies installed successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error during installation of core dependencies: {e}")
    print("Please check pip logs for details. Critical dependencies might be missing or incompatible.")
    sys.exit(1)

print("Dependency setup complete. You may now proceed with the rest of the notebook.")

Starting consolidated dependency installation...
Upgrading pip...
pip upgraded successfully.
Clearing pip cache...
Pip cache cleared.
Installing albumentations==2.0.8, grad-cam==1.5.5, ttach, opencv-python, matplotlib, gradio, rembg[cpu]...
All specified dependencies installed successfully.
Dependency setup complete. You may now proceed with the rest of the notebook.


In [3]:
import subprocess
import sys

# Reinstall torch and torchvision to resolve potential numpy binary incompatibility.
# This is often necessary in environments like Colab where numpy might get updated
# by other packages, leading to conflicts with pre-compiled torchvision.
print("Attempting to reinstall torch and torchvision to resolve potential numpy conflicts...")
try:
    # Use --force-reinstall to ensure they pick up the current numpy environment.
    # Use --upgrade to get the latest compatible versions.
    # The `--index-url` ensures we fetch the correct CUDA-enabled versions for Colab's GPU.
    # (Assuming CUDA 12.1, which is common for Colab's T4 GPUs. Adjust if needed for a different CUDA version).
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "--upgrade", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("torch and torchvision reinstalled successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error reinstalling torch and torchvision: {e}")
    print("Continuing, but expect potential issues if the conflict is not resolved.")

Attempting to reinstall torch and torchvision to resolve potential numpy conflicts...
torch and torchvision reinstalled successfully.


In [4]:
import os

# Define the local SSD target directory
LOCAL_DATA_DIR = '/tmp/Datasets_367v_Cleaned_224'

# Create the target directory if it doesn't exist
!mkdir -p {LOCAL_DATA_DIR}

# Copy the dataset from Google Drive to local SSD
# The -r flag ensures recursive copying for directories
# The -L flag (or --dereference) will follow symbolic links, if any
# Using `rsync -av` is generally more robust for large directories if available
print(f"Copying dataset from Google Drive to {LOCAL_DATA_DIR}...")
# Reverting to copy the *contents* of Datasets_3c into LOCAL_DATA_DIR
!cp -r /content/drive/MyDrive/Datasets_367v_Cleaned_224/* {LOCAL_DATA_DIR}/
print("Dataset copied to local SSD.")

print(f"Contents of {LOCAL_DATA_DIR} after copy:")
!ls -F {LOCAL_DATA_DIR}

Copying dataset from Google Drive to /tmp/Datasets_367v_Cleaned_224...
Dataset copied to local SSD.
Contents of /tmp/Datasets_367v_Cleaned_224 after copy:
01_Train/  02_Val/  03_Test/


In [5]:
# JUST CHECKING
print(f"Contents of Google Drive source directory: /content/drive/MyDrive/Datasets_367v_Cleaned_224")
!ls -F /content/drive/MyDrive/Datasets_367v_Cleaned_224

Contents of Google Drive source directory: /content/drive/MyDrive/Datasets_367v_Cleaned_224
01_Train/  02_Val/  03_Test/


In [6]:
import os

# Define the local SSD target directory (assuming LOCAL_DATA_DIR is defined in cell 9c82739e)
LOCAL_DATA_DIR = '/tmp/Datasets_367v_Cleaned_224'

# Update directory paths to point to the original local SSD dataset
TRAIN_DIR = os.path.join(LOCAL_DATA_DIR, '01_Train')
VAL_DIR = os.path.join(LOCAL_DATA_DIR, '02_Val')
TEST_DIR = os.path.join(LOCAL_DATA_DIR, '03_Test')

print(f"Updated TRAIN_DIR: {TRAIN_DIR}")
print(f"Updated VAL_DIR: {VAL_DIR}")
print(f"Updated TEST_DIR: {TEST_DIR}")

Updated TRAIN_DIR: /tmp/Datasets_367v_Cleaned_224/01_Train
Updated VAL_DIR: /tmp/Datasets_367v_Cleaned_224/02_Val
Updated TEST_DIR: /tmp/Datasets_367v_Cleaned_224/03_Test


In [7]:
import sys

import os
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v3_large

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

  # **#         END OF DATA PREPROCESSING**